<a href="https://colab.research.google.com/github/IrinaBekh/rec_sys/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

print("Генерация фиктивных данных для рекомендательной системы...")

# Генерация синтетических данных
np.random.seed(42)
num_samples = 500

data = {
    'age': np.random.randint(18, 65, num_samples),
    'country': np.random.choice(['USA', 'UK', 'Germany', 'France', 'Japan'], num_samples),
    'genre_preference': np.random.choice(['Action', 'Comedy', 'Drama', 'SciFi', 'Thriller'], num_samples)
}
df_users = pd.DataFrame(data)

# Фиктивная логика рекомендации: определяем 'рекомендуемый фильм' на основе признаков
# Это очень упрощенная логика для демонстрации
def assign_movie(row):
    if row['age'] < 30 and row['genre_preference'] == 'Action':
        return 'Movie A: Young Action'
    elif row['age'] >= 30 and row['country'] == 'USA' and row['genre_preference'] == 'Drama':
        return 'Movie B: US Drama'
    elif row['country'] == 'Germany' and row['genre_preference'] == 'SciFi':
        return 'Movie C: German SciFi'
    elif row['genre_preference'] == 'Comedy':
        return 'Movie D: General Comedy'
    else:
        return 'Movie E: Other'

df_users['recommended_movie'] = df_users.apply(assign_movie, axis=1)

print("Данные сгенерированы. Первые 5 строк:")
display(df_users.head())


Генерация фиктивных данных для рекомендательной системы...
Данные сгенерированы. Первые 5 строк:


,age,country,genre_preference,recommended_movie
0,56,USA,Comedy,Movie D: General Comedy
1,46,Japan,Thriller,Movie E: Other
2,32,Germany,Drama,Movie E: Other
3,60,Germany,Drama,Movie E: Other
4,25,Japan,Drama,Movie E: Other


In [2]:
print("Подготовка данных и обучение модели...")

# Разделение признаков (X) и целевой переменной (y)
X = df_users[['age', 'country', 'genre_preference']]
y = df_users['recommended_movie']

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Создание предобработчика для категориальных признаков
# OneHotEncoder для 'country' и 'genre_preference'
categorical_features = ['country', 'genre_preference']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # 'age' будет оставлен как есть
)

# Создание и обучение пайплайна модели (препроцессор + Decision Tree Classifier)
model_rec = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

model_rec.fit(X_train, y_train)

print("Модель рекомендательной системы успешно обучена.")

# Оценка модели
accuracy_rec = model_rec.score(X_test, y_test)
print(f"Точность модели на тестовых данных: {accuracy_rec:.2f}")


Подготовка данных и обучение модели...
Модель рекомендательной системы успешно обучена.
Точность модели на тестовых данных: 0.99


In [3]:
print("Демонстрация получения рекомендации для нового пользователя:")

# Пример нового пользователя
new_user_data = pd.DataFrame([{
    'age': 25,
    'country': 'USA',
    'genre_preference': 'Action'
}])

# Получение рекомендации
recommendation = model_rec.predict(new_user_data)
print(f"Для пользователя с данными {new_user_data.iloc[0].to_dict()}, рекомендуется фильм: {recommendation[0]}")

new_user_data_2 = pd.DataFrame([{
    'age': 45,
    'country': 'Germany',
    'genre_preference': 'SciFi'
}])
recommendation_2 = model_rec.predict(new_user_data_2)
print(f"Для пользователя с данными {new_user_data_2.iloc[0].to_dict()}, рекомендуется фильм: {recommendation_2[0]}")

new_user_data_3 = pd.DataFrame([{
    'age': 32,
    'country': 'UK',
    'genre_preference': 'Comedy'
}])
recommendation_3 = model_rec.predict(new_user_data_3)
print(f"Для пользователя с данными {new_user_data_3.iloc[0].to_dict()}, рекомендуется фильм: {recommendation_3[0]}")


Демонстрация получения рекомендации для нового пользователя:
Для пользователя с данными {'age': 25, 'country': 'USA', 'genre_preference': 'Action'}, рекомендуется фильм: Movie A: Young Action
Для пользователя с данными {'age': 45, 'country': 'Germany', 'genre_preference': 'SciFi'}, рекомендуется фильм: Movie C: German SciFi
Для пользователя с данными {'age': 32, 'country': 'UK', 'genre_preference': 'Comedy'}, рекомендуется фильм: Movie D: General Comedy


In [4]:
import pickle

with open('model_rec.pkl', 'wb') as f:
  pickle.dump(model_rec, f)

# Инференс

In [5]:
import requests

# Замените этот URL на ваш текущий публичный URL от ngrok
NGROK_PUBLIC_URL = "https://discontented-dusty-leeriest.ngrok-free.dev"

# Пример данных для предсказания
payload = {
    "age": 30,
    "country": "USA",
    "genre_preference": "Comedy"
}

# Отправка POST-запроса
response = requests.post(f"{NGROK_PUBLIC_URL}/prediction", json=payload)

# Вывод результата
if response.status_code == 200:
    print("Успешный POST-запрос:")
    print(response.json())
else:
    print(f"Ошибка при выполнении POST-запроса: {response.status_code}")
    print(response.text)


Ошибка при выполнении POST-запроса: 404
<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin

In [6]:
import requests

# Замените этот URL на ваш текущий публичный URL от ngrok
NGROK_PUBLIC_URL = "https://discontented-dusty-leeriest.ngrok-free.dev"

# Пример тестирования GET-запроса для здоровья сервиса
health_response = requests.get(f"{NGROK_PUBLIC_URL}/health")

if health_response.status_code == 200:
    print("Статус сервиса (GET /health):")
    print(health_response.json())
else:
    print(f"Ошибка при запросе /health: {health_response.status_code}")
    print(health_response.text)

# Пример тестирования GET-запроса для статистики
stats_response = requests.get(f"{NGROK_PUBLIC_URL}/stats")

if stats_response.status_code == 200:
    print("Статистика запросов (GET /stats):")
    print(stats_response.json())
else:
    print(f"Ошибка при запросе /stats: {stats_response.status_code}")
    print(stats_response.text)


Ошибка при запросе /health: 404
<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonym

In [7]:
import pandas as pd

new_data = pd.DataFrame([{
    'age': 35,
    'country': 'France',
    'genre_preference': 'Drama'
}])

prediction = model_rec.predict(new_data)
print(f'Рекомендуем Вам фильм: {prediction}')

Рекомендуем Вам фильм: ['Movie E: Other']


# ТЕСТ API
get - постучались - получили ответ

post - постучались, отправили данные, получили ответ на основе данных

In [8]:
from fastapi import FastAPI, Request, HTTPException
from pydantic import BaseModel
import pandas as pd
import pickle

app = FastAPI()
with open('model_rec.pkl', 'rb') as f:
    model = pickle.load(f)

request_count = 0

class PredictionInput(BaseModel):
    age: int
    country: str
    genre_preference: str

@app.get('/stats')
def stats():
    return {'request_count': request_count}

@app.get('/health')
def health():
    return {'status': "OK"}

@app.post('/prediction')
def prediction(data: PredictionInput):
    global request_count
    request_count += 1

    new_data = pd.DataFrame({
        'age': [data.age],
        'country': [data.country],
        'genre_preference': [data.genre_preference]
    })
    prediction_result = model.predict(new_data)
    return {'Рекомендуем Вам:': prediction_result.tolist()}

# Task
To deploy your FastAPI application in Google Colab, you need to set up `pyngrok` to create a public URL. First, install the `pyngrok` library. After installation, restart the runtime (`Runtime -> Restart runtime`) and then run all cells (`Runtime -> Run all`) to ensure `nest_asyncio` is correctly applied. Finally, modify the `uvicorn` cell to use `pyngrok` to expose your FastAPI service and provide you with a public URL to access and test your API.

## Install Pyngrok

### Subtask:
Установите библиотеку `pyngrok`, которая позволит создать публичный URL для вашего FastAPI приложения.


**Reasoning**:
The subtask requires installing the `pyngrok` library. I will use `pip install` in a code block to accomplish this.



In [9]:
get_ipython().system('pip install pyngrok')
print("Pyngrok установлен. Пожалуйста, перезапустите среду выполнения (Runtime -> Restart runtime) и затем запустите все ячейки (Runtime -> Run all). После этого мы модифицируем ячейку Uvicorn для использования Pyngrok.")

Pyngrok установлен. Пожалуйста, перезапустите среду выполнения (Runtime -> Restart runtime) и затем запустите все ячейки (Runtime -> Run all). После этого мы модифицируем ячейку Uvicorn для использования Pyngrok.


In [10]:
if __name__ == '__main__':
    import uvicorn
    import nest_asyncio
    import threading
    import time
    from pyngrok import ngrok

    nest_asyncio.apply()

    ngrok.set_auth_token("3AGMD6A6jFlnt7BaIGloJwKHz0A_77J66ExzmR4Ks4LSupvrx")

    # Kill any existing ngrok tunnels
    ngrok.kill()

    # Start ngrok tunnel
    public_url = ngrok.connect(5000)
    print(f"Public URL: {public_url}")

    # Function to run Uvicorn in a separate thread
    def run_uvicorn():
        uvicorn.run(app, host='0.0.0.0', port=5000, log_level="warning")

    # Start Uvicorn in a new thread
    thread = threading.Thread(target=run_uvicorn)
    thread.start()

    print("FastAPI приложение запущено в фоновом режиме. Используйте публичный URL выше для доступа.")

Public URL: NgrokTunnel: "https://discontented-dusty-leeriest.ngrok-free.dev" -> "http://localhost:5000"
FastAPI приложение запущено в фоновом режиме. Используйте публичный URL выше для доступа.


In [11]:
import requests

def predict_model(data):
    url = 'https://discontented-dusty-leeriest.ngrok-free.dev/prediction' # Corrected URL
    response = requests.post(url, json=data)
    if response.status_code == 200:
       return response.json()
    else:
        return f"Ошибка: {response.status_code}, {response.text}"
data = {
    'age': 20,
    'country': 'USA',
    'genre_preference': 'Comedy'

}
prediction = predict_model(data)
print(prediction)

Ошибка: 502, <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link r